In [ ]:
import jax
import jax.numpy as jnp
from jax import random
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import plotly.express as px

In [ ]:
from src.jax_resnet.model import init_params, FiniteResNetParams, relu, batched_forward_track
from src.jax_resnet.training import train_scan_jit, train_dropout_scan_jit, train_ram_scan_jit, train_scan_ce_jit, train_dropout_scan_ce_jit, train_ram_scan_ce_jit

In [ ]:
from src.utils import make_dataset_mnist, align_tracked_particle_across_layers

In [ ]:
d_in, d_out, seed = 784, 2, 42
N=100 # subsample of MNIST to use in the experiment (this affects the training time since we track the loss on the full test set over training iterations).
X_train, Y_train, X_test, Y_test = make_dataset_mnist(N=N, seed=seed, digits=[4,7])

In [ ]:
X_train.shape, Y_train.shape, X_test.shape, Y_test.shape

In [ ]:
D, L, M = 10, 4, 8 # These can be modified at will.
tau = 0.4
n_steps = 200
lr_in, lr_out = 0.0, 0.0
q = 0.5
batch_size = 64
last_particle_single_source = True #Is the mask also shared in depth for the last particle?

In [ ]:
params0 = init_params(random.PRNGKey(seed + 44), d_in, d_out, D, L, M)
params0 = align_tracked_particle_across_layers(params0, particle_idx=-1)

In [ ]:
general_kwargs = dict(X_train=X_train, Y_train=Y_train, X_test=X_test, Y_test=Y_test, 
                 lr=tau, lr_in=lr_in, lr_out=lr_out, n_steps=n_steps, 
                 batch_size=batch_size, track_outputs=False, activation=relu)

In [ ]:
kwargs_do = general_kwargs | dict(q_layers=(q * jnp.ones(L)), q_in=1.0, q_out=1.0, 
                 single_source_last_particle=last_particle_single_source)
variants = [
    "full_unit_dropout",
    "stochastic_depth",
    "single_source_M",
]

In [ ]:
def loop_experiment(num_repetitions, loop_seed, train_fn, params0, kwargs, variants, dropout = True):
    final_params = {}
    histories = {}
    for variant in variants:
        final_params_repeats = []
        histories_repeats = []
        for rep in range(num_repetitions):
            if dropout:
                fp, his = train_fn(
                    params0, 
                    internal_dropout_variant=variant, 
                    key=random.PRNGKey(loop_seed+rep),
                    **kwargs)
            else:
                fp, his = train_fn(
                    params0, 
                    key=random.PRNGKey(loop_seed+rep),
                    **kwargs)
                
            _ = jax.block_until_ready(his)
            final_params_repeats.append(fp)
            histories_repeats.append(his)
        final_params[variant] = final_params_repeats
        histories[variant] = histories_repeats
    return final_params, histories

In [ ]:
num_repetitions = 5
LOOP_SEED = 48

In [ ]:
final_params_gd, histories_gd = loop_experiment(num_repetitions, LOOP_SEED, train_scan_ce_jit, params0, general_kwargs, ["gd"], dropout=False)

In [ ]:
final_params_do, histories_do = loop_experiment(num_repetitions, LOOP_SEED, train_dropout_scan_ce_jit, params0, kwargs_do, variants, dropout=True)

In [ ]:
final_params_ram, histories_ram = loop_experiment(num_repetitions, LOOP_SEED, train_ram_scan_ce_jit, params0, kwargs_do, variants, dropout=True)

In [ ]:
def compute_train_h_out(params):
    th, ty = batched_forward_track(params, X_test, activation=relu)
    return th, ty

def add_train_h_out(params_dict, history_dict):
    new_dict = history_dict.copy()
    for variant in params_dict.keys():
        for j in range(len(params_dict[variant])):
            th, ty = compute_train_h_out(params_dict[variant][j])
            new_dict[variant][j]["test_h"] = th
            new_dict[variant][j]["test_output"] = ty
    return new_dict

In [ ]:
histories_do = add_train_h_out(final_params_do, histories_do)
histories_ram = add_train_h_out(final_params_ram, histories_ram)
histories_gd = add_train_h_out(final_params_gd, histories_gd)

### Save results from this run

In [ ]:
import pickle


setting_str = f'L{L}_M{M}_D{D}_tau{tau}_q{q}_nsteps{n_steps}_din{d_in}_dout{d_out}_seed{seed}_N{N}_numrepetitions{num_repetitions}_loopseed{LOOP_SEED}_batchsize{batch_size}_lpss{last_particle_single_source}'


# Export (serialize) to pickle
with open(f'data/mnist/final_params_gd_{setting_str}.pkl', 'wb') as f:   # 'wb' = write binary
    pickle.dump(final_params_gd, f)

with open(f'data/mnist/histories_gd_{setting_str}.pkl', 'wb') as f:   # 'wb' = write binary
    pickle.dump(histories_gd, f)

with open(f'data/mnist/final_params_do_{setting_str}.pkl', 'wb') as f:   # 'wb' = write binary
    pickle.dump(final_params_do, f)

with open(f'data/mnist/histories_do_{setting_str}.pkl', 'wb') as f:   # 'wb' = write binary
    pickle.dump(histories_do, f)

with open(f'data/mnist/final_params_ram_{setting_str}.pkl', 'wb') as f:   # 'wb' = write binary
    pickle.dump(final_params_ram, f)

with open(f'data/mnist/histories_ram_{setting_str}.pkl', 'wb') as f:   # 'wb' = write binary
    pickle.dump(histories_ram, f)
